In [5]:
from lassi.gprof import *

In [6]:
gprofile = GProf(Path("/home/gbrun/TEST/matmul.c"))

In [7]:
gprofile.profile(args="1000 1000 1000", type="flat")

Compiling /home/gbrun/TEST/matmul.c using gcc...
Compiling with command: gcc /home/gbrun/TEST/matmul.c -pg -no-pie -fno-builtin -o /home/gbrun/TEST/matmul_gprof.out


Running with command: /home/gbrun/TEST/matmul_gprof.out 1000 1000 1000


Profiling with gprof: gprof /home/gbrun/TEST/matmul_gprof.out gmon.out -p


'| Function Name | % Time | Self (s) | Calls | ms/call |\n| :--- | :---: | :---: | :---: | :---: |\n| **matrix_multiply** | 100.0% | 5.98s | 1 | 5.98 |\n| init_matrices | 0.0% | 0.0s | 1 | 0.0 |'

In [9]:
def parse_to_memory_schema(gprof_output: str):
    """
    Parses gprof output and converts it directly into 
    MCP Memory Server compatible entities and relations.
    """
    blocks = re.split(r'-{10,}', gprof_output)
    
    # We use a dict to deduplicate entities by name
    entities_map = {}
    relations = []

    for block in blocks:
        lines = block.strip().split('\n')
        if not lines: continue

        # 1. Identify the Focus Node (The line starting with [N])
        focus_idx = -1
        for i, line in enumerate(lines):
            if re.match(r'^\s*\[(\d+)\]', line):
                focus_idx = i
                break
        
        if focus_idx == -1: continue

        # 2. Parse the Focus Node
        # Regex: [index] %time self children name
        focus_match = re.search(r'\[(\d+)\]\s+([\d\.]+)?\s*([\d\.]+)?\s*([\d\.]+)?\s+(?:[\d/]+)?\s+([^\s]+)', lines[focus_idx])
        
        if focus_match:
            idx, pct, self_t, child_t, name = focus_match.groups()
            
            # Create the Entity observations
            # We treat performance metrics as textual "observations" for the memory graph
            obs = []
            if pct: obs.append(f"consumes {pct}% of total time")
            if self_t: obs.append(f"self execution time: {self_t}s")
            if child_t: obs.append(f"children execution time: {child_t}s")
            
            entities_map[name] = {
                "name": name,
                "kind": "Function",
                "observations": obs
            }
            
            # 3. Parse Callers (Incoming Edges)
            for i in range(0, focus_idx):
                line = lines[i].strip()
                if "<spontaneous>" in line:
                    # Create a node for the OS/Runtime if it doesn't exist
                    if "OS_Runtime" not in entities_map:
                        entities_map["OS_Runtime"] = {
                            "name": "OS_Runtime", 
                            "kind": "System", 
                            "observations": ["The operating system or runtime environment"]
                        }
                    relations.append({
                        "from": "OS_Runtime",
                        "to": name,
                        "relationType": "spawns"
                    })
                else:
                    parts = line.split()
                    if len(parts) >= 2:
                        # Extract caller name (heuristic: 2nd to last item usually)
                        caller = parts[-2] if parts[-1].startswith('[') else parts[-1]
                        
                        # Ensure caller exists as an entity (placeholder until fully parsed)
                        if caller not in entities_map:
                            entities_map[caller] = {"name": caller, "kind": "Function", "observations": []}
                            
                        relations.append({
                            "from": caller,
                            "to": name,
                            "relationType": "calls"
                        })

            # 4. Parse Callees (Outgoing Edges)
            for i in range(focus_idx + 1, len(lines)):
                line = lines[i].strip()
                parts = line.split()
                if len(parts) >= 2:
                    callee = parts[-2] if parts[-1].startswith('[') else parts[-1]
                    
                    if callee not in entities_map:
                        entities_map[callee] = {"name": callee, "kind": "Function", "observations": []}
                        
                    relations.append({
                        "from": name,
                        "to": callee,
                        "relationType": "calls"
                    })

    return list(entities_map.values()), relations

def parse_profile_for_memory(profile_text: str) -> dict:
    """
    Parses a gprof profile and prepares data for the 'memory' server.
    Returns a JSON object with 'entities' and 'relations' ready to be stored.
    """
    entities, relations = parse_to_memory_schema(profile_text)
    
    return {
        "status": "success",
        "instruction_to_model": "Please call create_entities and create_relations on the 'memory' server using the data below.",
        "data": {
            "entities": entities,
            "relations": relations
        }
    }

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.toolsets.fastmcp import FastMCPToolset

toolset = FastMCPToolset()

In [10]:
parse_to_memory_schema(
    """Call graph (explanation follows)


granularity: each sample hit covers 4 byte(s) for 0.21% of 4.85 seconds

index % time    self  children    called     name
                4.85    0.00       1/1           main [2]
[1]    100.0    4.85    0.00       1         matrix_multiply [1]
-----------------------------------------------
                                                 <spontaneous>
[2]    100.0    0.00    4.85                 main [2]
                4.85    0.00       1/1           matrix_multiply [1]
                0.00    0.00       1/1           init_matrices [3]
-----------------------------------------------
                0.00    0.00       1/1           main [2]
[3]      0.0    0.00    0.00       1         init_matrices [3]
-----------------------------------------------
)""")

([{'name': 'matrix_multiply',
   'kind': 'Function',
   'observations': ['consumes 100.0% of total time',
    'self execution time: 4.85s',
    'children execution time: 0.00s']},
  {'name': 'follows)', 'kind': 'Function', 'observations': []},
  {'name': 'seconds', 'kind': 'Function', 'observations': []},
  {'name': 'name', 'kind': 'Function', 'observations': []},
  {'name': 'main',
   'kind': 'Function',
   'observations': ['consumes 100.0% of total time',
    'self execution time: 0.00s',
    'children execution time: 4.85s']},
  {'name': 'OS_Runtime',
   'kind': 'System',
   'observations': ['The operating system or runtime environment']},
  {'name': 'init_matrices',
   'kind': 'Function',
   'observations': ['consumes 0.0% of total time',
    'self execution time: 0.00s',
    'children execution time: 0.00s']}],
 [{'from': 'follows)', 'to': 'matrix_multiply', 'relationType': 'calls'},
  {'from': 'seconds', 'to': 'matrix_multiply', 'relationType': 'calls'},
  {'from': 'name', 'to': 

In [ ]:
{
  "entities": [
    {
      "observations": [
        "consumes 100.0% of total time",
        "self execution time: 4.85s",
        "children execution time: 0.00s"
      ],
      "name": "matrix_multiply",
      "entityType": "Function"
    },
    {
      "observations": [],
      "entityType": "Function",
      "name": "follows)"
    },
    {
      "entityType": "Function",
      "observations": [],
      "name": "seconds"
    },
    {
      "name": "name",
      "entityType": "Function",
      "observations": []
    },
    {
      "observations": [
        "consumes 100.0% of total time",
        "self execution time: 0.00s",
        "children execution time: 4.85s"
      ],
      "name": "main",
      "entityType": "Function"
    },
    {
      "observations": [
        "The operating system or runtime environment"
      ],
      "name": "OS_Runtime",
      "entityType": "System"
    },
    {
      "name": "init_matrices",
      "observations": [
        "consumes 0.0% of total time",
        "self execution time: 0.00s",
        "children execution time: 0.00s"
      ],
      "entityType": "Function"
    }
  ]
}

In [ ]:
{
  "relations": [
    {
      "to": "matrix_multiply",
      "from": "follows)",
      "relationType": "calls"
    },
    {
      "relationType": "calls",
      "from": "seconds",
      "to": "matrix_multiply"
    },
    {
      "from": "name",
      "relationType": "calls",
      "to": "matrix_multiply"
    },
    {
      "to": "matrix_multiply",
      "from": "main",
      "relationType": "calls"
    },
    {
      "from": "OS_Runtime",
      "relationType": "spawns",
      "to": "main"
    },
    {
      "relationType": "calls",
      "from": "main",
      "to": "init_matrices"
    }
  ]
}